In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

# ── Load data ──────────────────────────────────────────
df = pd.read_csv('data/raw/ckd.csv')
print("Loaded shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)


Loaded shape: (500, 12)

Missing values:
 age                       0
donor_age                 0
hla_mismatch              0
cold_ischemia_hours       0
pra                       0
dialysis_years            0
living_donor              0
egfr_1year                0
delayed_graft_function    0
donor_hypertension        0
duration_years            0
graft_failure             0
dtype: int64

Data types:
 age                         int64
donor_age                   int64
hla_mismatch                int64
cold_ischemia_hours       float64
pra                       float64
dialysis_years            float64
living_donor                int64
egfr_1year                float64
delayed_graft_function      int64
donor_hypertension          int64
duration_years            float64
graft_failure               int64
dtype: object


In [2]:
# ── Feature Engineering ────────────────────────────────

# 1. Donor Risk Index (DRI) — combines donor age + hypertension
df['donor_risk_index'] = (
    0.4 * (df['donor_age'] / 70) +
    0.6 * df['donor_hypertension']
).round(3)

# 2. Recipient age group
df['age_group'] = pd.cut(df['age'],
    bins=[0, 35, 50, 65, 100],
    labels=['young', 'middle', 'senior', 'elderly'])

# 3. Cold ischemia risk category
df['ischemia_risk'] = pd.cut(df['cold_ischemia_hours'],
    bins=[0, 12, 24, 100],
    labels=['low', 'medium', 'high'])

# 4. Immunological risk (HLA + PRA combined)
df['immune_risk_score'] = (
    0.5 * df['hla_mismatch'] / 6 +
    0.5 * df['pra'] / 100
).round(3)

# 5. eGFR category (kidney function at 1 year)
df['egfr_category'] = pd.cut(df['egfr_1year'],
    bins=[0, 30, 45, 60, 100],
    labels=['poor', 'moderate', 'good', 'excellent'])

print("New features added:")
print(df[['donor_risk_index', 'age_group', 
          'ischemia_risk', 'immune_risk_score', 
          'egfr_category']].head(8))
print("\nShape after engineering:", df.shape)

New features added:
   donor_risk_index age_group ischemia_risk  immune_risk_score egfr_category
0             0.297    senior           low              0.190      moderate
1             0.926   elderly          high              0.470          poor
2             0.394    middle          high              0.512          poor
3             0.189     young          high              0.517          poor
4             0.771    senior          high              0.891          poor
5             0.383     young          high              0.852          poor
6             0.337    middle        medium              0.723          poor
7             0.269    senior          high              0.849          poor

Shape after engineering: (500, 17)


In [8]:
# ── Encode categorical features ────────────────────────
df_model = df.copy()

# Convert categories to numbers
df_model['age_group'] = df_model['age_group'].map(
    {'young': 0, 'middle': 1, 'senior': 2, 'elderly': 3})

df_model['ischemia_risk'] = df_model['ischemia_risk'].map(
    {'low': 0, 'medium': 1, 'high': 2})

df_model['egfr_category'] = df_model['egfr_category'].map(
    {'poor': 0, 'moderate': 1, 'good': 2, 'excellent': 3})

# ── Define features and target ─────────────────────────
FEATURES = [
    'age', 'donor_age', 'hla_mismatch', 'cold_ischemia_hours',
    'pra', 'dialysis_years', 'living_donor', 'egfr_1year',
    'delayed_graft_function', 'donor_hypertension',
    'donor_risk_index', 'immune_risk_score',
    'age_group', 'ischemia_risk', 'egfr_category'
]

X = df_model[FEATURES]
y_duration = df_model['duration_years']
y_event    = df_model['graft_failure']

# ── Train / Test split ─────────────────────────────────
X_train, X_test, dur_train, dur_test, evt_train, evt_test = train_test_split(
    X, y_duration, y_event,
    test_size=0.2, random_state=42
)

print(f"Training set:  {X_train.shape[0]} patients")
print(f"Test set:      {X_test.shape[0]} patients")
print(f"\nFailure rate - train: {evt_train.mean()*100:.1f}%")
print(f"Failure rate - test:  {evt_test.mean()*100:.1f}%")

# ── Scale features ─────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURES)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=FEATURES)

print("\n✅ Features scaled")
print("Sample scaled values:\n", X_train_scaled.head(3).round(3))

Training set:  400 patients
Test set:      100 patients

Failure rate - train: 53.2%
Failure rate - test:  49.0%

✅ Features scaled
Sample scaled values:
      age  donor_age  hla_mismatch  cold_ischemia_hours    pra  dialysis_years  \
0 -0.634     -0.455        -0.053               -0.652 -1.297           0.090   
1 -1.718     -0.658        -1.030               -0.091  0.469           0.588   
2  1.532     -1.744         1.414                0.161 -1.350          -1.053   

   living_donor  egfr_1year  delayed_graft_function  donor_hypertension  \
0        -0.791       0.812                  -0.585               1.106   
1         1.264       1.734                  -0.585              -0.905   
2        -0.791      -1.077                   1.709               1.106   

   donor_risk_index  immune_risk_score  age_group  ischemia_risk  \
0             0.968             -0.917     -0.372         -0.358   
1            -1.083             -0.525     -1.313         -0.358   
2             0

In [9]:
# ── Save cleaned data ──────────────────────────────────
os.makedirs('data/clean', exist_ok=True)

# Full cleaned dataset
df_model.to_csv('data/clean/ckd_clean.csv', index=False)

# Train and test sets
X_train_scaled['duration_years'] = dur_train.values
X_train_scaled['graft_failure']  = evt_train.values
X_train_scaled.to_csv('data/clean/train.csv', index=False)

X_test_scaled['duration_years'] = dur_test.values
X_test_scaled['graft_failure']  = evt_test.values
X_test_scaled.to_csv('data/clean/test.csv', index=False)

# Save the scaler for later use in the web app
joblib.dump(scaler, 'models/scaler.pkl')

print("✅ Files saved:")
print("   data/clean/ckd_clean.csv  — full cleaned dataset (500 patients)")
print("   data/clean/train.csv      — 400 patients for training")
print("   data/clean/test.csv       — 100 patients for testing")
print("   models/scaler.pkl         — scaler for the web app")

✅ Files saved:
   data/clean/ckd_clean.csv  — full cleaned dataset (500 patients)
   data/clean/train.csv      — 400 patients for training
   data/clean/test.csv       — 100 patients for testing
   models/scaler.pkl         — scaler for the web app


In [10]:
train = pd.read_csv('data/clean/train.csv')
print("Shape:", train.shape)
print("\nColumns:", list(train.columns))
print("\nFirst 3 rows:")
print(train.head(3).round(3))
print("\nStats:")
print(train.describe().round(2))

Shape: (400, 17)

Columns: ['age', 'donor_age', 'hla_mismatch', 'cold_ischemia_hours', 'pra', 'dialysis_years', 'living_donor', 'egfr_1year', 'delayed_graft_function', 'donor_hypertension', 'donor_risk_index', 'immune_risk_score', 'age_group', 'ischemia_risk', 'egfr_category', 'duration_years', 'graft_failure']

First 3 rows:
     age  donor_age  hla_mismatch  cold_ischemia_hours    pra  dialysis_years  \
0 -0.634     -0.455        -0.053               -0.652 -1.297           0.090   
1 -1.718     -0.658        -1.030               -0.091  0.469           0.588   
2  1.532     -1.744         1.414                0.161 -1.350          -1.053   

   living_donor  egfr_1year  delayed_graft_function  donor_hypertension  \
0        -0.791       0.812                  -0.585               1.106   
1         1.264       1.734                  -0.585              -0.905   
2        -0.791      -1.077                   1.709               1.106   

   donor_risk_index  immune_risk_score  age_gr